# Modera Reynoldstown Apartment Monitor

This notebook checks the Modera Reynoldstown website for **7th-floor 2-bed or 3-bed** units and emails you when it finds them.

**You only have to do four things, in order:**

1. Get a Gmail App Password (one time, 2 minutes — instructions below).
2. Paste it into **Cell 1 (CONFIG)** below.
3. Click **Runtime → Run all** at the top of the page.
4. (Optional) Run the last cell to keep it auto-checking every few hours while this tab stays open.

That's it. You'll get an email after Cell 3 finishes — even on the first run.

---
## How to get your Gmail App Password (one-time, 2 minutes)

Gmail won't accept your regular password from a script — you need a special 16-character "App Password."

1. Open this link in a new tab: <https://myaccount.google.com/security>
2. Under **"How you sign in to Google"**, turn on **2-Step Verification** if it's off. Follow the prompts (phone number + code).
3. Open this link: <https://myaccount.google.com/apppasswords>
4. In the **App name** box, type `Apartment Monitor`, then click **Create**.
5. Google shows a 16-character password like `abcd efgh ijkl mnop`. **Copy it now** (you can't see it again later).
6. Paste it into the cell directly below, into the `GMAIL_APP_PASSWORD` line.

## Cell 1 — CONFIG (edit these three lines, then run the cell)

Replace the example values with yours. Keep the quotes.

In [ ]:
# ====== EDIT THESE THREE VALUES ======

GMAIL_EMAIL        = "your.address@gmail.com"       # the Gmail account that will SEND the emails
GMAIL_APP_PASSWORD = "abcd efgh ijkl mnop"          # the 16-character App Password from Google (spaces OK)
RECIPIENTS         = [
    "Omid.razmpour@emory.edu",
    "dcginouves@gmail.com",
]

# ====== DO NOT EDIT BELOW THIS LINE ======

TARGET_FLOOR    = 7        # 7th floor only
TARGET_BEDROOMS = {2, 3}   # 2-bed or 3-bed only
LISTINGS_URL    = "https://www.moderareynoldstown.com/atlanta/modera-reynoldstown/conventional/"

assert "@" in GMAIL_EMAIL,                       "Set GMAIL_EMAIL to your Gmail address"
assert GMAIL_APP_PASSWORD != "abcd efgh ijkl mnop", "Paste your real Gmail App Password"
assert all("@" in r for r in RECIPIENTS),        "Recipient list looks wrong"
print("Config looks good. From:", GMAIL_EMAIL, "| To:", RECIPIENTS)

## Cell 2 — Connect Google Drive (so the notebook remembers which units it already emailed about)

When you run this cell, Colab will pop up a window asking you to sign in to Google and grant access. Click through and accept — this lets the script save a tiny `state.json` file to your Drive so it doesn't re-email you about the same unit twice.

> Don't want to use Drive? Skip this cell. The script will still work; it just won't remember between runs, so you might get duplicate notifications.

In [ ]:
import os, json
from pathlib import Path

STATE_DIR = Path("/content")  # default: in-memory only
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    STATE_DIR = Path("/content/drive/MyDrive/apartment_monitor")
    STATE_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive connected. State will be saved at:", STATE_DIR / "state.json")
except Exception as e:
    print("Drive NOT connected (", e, ") — state will only last this session.")

STATE_PATH = STATE_DIR / "state.json"
LOG_PATH   = STATE_DIR / "monitor.log"
print("State file:", STATE_PATH)

## Cell 3 — Run the check and send the email

This is the cell that does the actual work. When you run it you'll see a few log lines, and within ~10 seconds an email will arrive at both inboxes (Omid's Emory address and dcginouves@gmail.com).

**Always sends an email**, even when no matching units are found, so you know the monitor is alive.

In [ ]:
import json, re, smtplib, traceback, logging, sys
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

import requests
from bs4 import BeautifulSoup

# --- logging (writes to monitor.log AND prints inline) ---
for h in list(logging.getLogger().handlers):
    logging.getLogger().removeHandler(h)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(str(LOG_PATH)), logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("apartment_monitor")

USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)
ENTRATA_API = "https://www.moderareynoldstown.com/api/v1/properties/modera-reynoldstown/availability"


def _normalize(raw):
    unit = raw.get("unitNumber") or raw.get("unit_number") or raw.get("unit") or raw.get("UnitNumber")
    beds = raw.get("bedrooms") or raw.get("beds") or raw.get("Bedrooms") or raw.get("bedroomCount")
    rent = raw.get("rent") or raw.get("price") or raw.get("startingRent") or raw.get("minRent")
    floor = raw.get("floor") or raw.get("Floor") or raw.get("floorNumber")
    avail = raw.get("availableDate") or raw.get("available_date") or raw.get("dateAvailable") or raw.get("availability")
    url   = raw.get("url") or raw.get("detailUrl") or raw.get("link")
    if unit is None or beds is None:
        return None
    try:
        beds = int(beds)
    except (TypeError, ValueError):
        return None
    if floor is None:
        digits = re.sub(r"\D", "", str(unit))
        if len(digits) >= 3:
            floor = int(digits[:-2])
    try:
        floor = int(floor) if floor is not None else None
    except (TypeError, ValueError):
        floor = None
    return {
        "unit": str(unit), "bedrooms": beds, "floor": floor,
        "rent": str(rent) if rent is not None else "N/A",
        "available": str(avail) if avail else "N/A",
        "url": url or LISTINGS_URL,
    }


def fetch_listings():
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT, "Accept": "application/json, text/html;q=0.9, */*;q=0.8"})
    # 1) try JSON API
    try:
        r = s.get(ENTRATA_API, timeout=30)
        if r.status_code == 200 and "json" in r.headers.get("content-type", ""):
            payload = r.json()
            rows = payload.get("units") if isinstance(payload, dict) else payload
            if isinstance(rows, list):
                out = [u for u in (_normalize(x) for x in rows) if u]
                if out:
                    log.info("Fetched %d units via API.", len(out))
                    return out
    except requests.RequestException as e:
        log.info("API unreachable (%s); falling back to HTML.", e)
    # 2) fall back to HTML
    r = s.get(LISTINGS_URL, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    out = []
    for row in soup.select("tr[data-unit], li[data-unit], div[data-unit]"):
        raw = {
            "unitNumber": row.get("data-unit"),
            "bedrooms":   row.get("data-beds") or row.get("data-bedrooms"),
            "rent":       row.get("data-rent") or row.get("data-price"),
            "floor":      row.get("data-floor"),
            "availableDate": row.get("data-available"),
        }
        link = row.find("a", href=True)
        if link:
            raw["url"] = link["href"]
        n = _normalize(raw)
        if n:
            out.append(n)
    if not out:
        for script in soup.find_all("script"):
            text = script.string or ""
            m = re.search(r"availability\s*[:=]\s*(\[.+?\])", text, re.DOTALL)
            if m:
                try:
                    rows = json.loads(m.group(1))
                except ValueError:
                    continue
                for x in rows:
                    n = _normalize(x)
                    if n:
                        out.append(n)
                if out:
                    break
    log.info("Fetched %d units via HTML scrape.", len(out))
    return out


def load_state():
    if STATE_PATH.exists():
        try:
            d = json.loads(STATE_PATH.read_text())
            d.setdefault("notified_units", [])
            return d
        except Exception:
            pass
    return {"notified_units": []}


def save_state(state):
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True))


def unit_key(u):
    return f"{u['unit']}|{u['bedrooms']}bd|floor{u['floor']}"


def build_email(new_units, all_matching):
    ts = datetime.now().strftime("%Y-%m-%d %I:%M %p")
    if new_units:
        subject = f"[Modera Reynoldstown] {len(new_units)} new 7th-floor 2/3-bed unit(s)"
        lines = [f"Check on {ts}", "", f"NEW matching units ({len(new_units)}):"]
        lines += [f"  - Unit {u['unit']}: {u['bedrooms']} bed, floor {u['floor']}, rent {u['rent']}, available {u['available']}, link: {u['url']}" for u in new_units]
        prev = [u for u in all_matching if u not in new_units]
        if prev:
            lines += ["", "Other matching units still listed:"]
            lines += [f"  - Unit {u['unit']}: {u['bedrooms']} bed, floor {u['floor']}, rent {u['rent']}, available {u['available']}" for u in prev]
        lines += ["", f"Source: {LISTINGS_URL}"]
        text = "\n".join(lines)
        items = "".join(f"<li><b>Unit {u['unit']}</b> — {u['bedrooms']} bed · floor {u['floor']} · {u['rent']} · avail {u['available']} (<a href=\"{u['url']}\">view</a>)</li>" for u in new_units)
        html  = f"<p>Check on {ts}</p><p><b>{len(new_units)} new matching unit(s):</b></p><ul>{items}</ul>"
        if prev:
            items2 = "".join(f"<li>Unit {u['unit']} — {u['bedrooms']} bed · floor {u['floor']} · {u['rent']}</li>" for u in prev)
            html += f"<p>Other matching units still listed:</p><ul>{items2}</ul>"
        html += f'<p>Source: <a href="{LISTINGS_URL}">{LISTINGS_URL}</a></p>'
    else:
        subject = "[Modera Reynoldstown] No new 7th-floor 2/3-bed units"
        text = f"Check on {ts}\n\nNo new 2 or 3 bed units on 7th floor currently available.\n\nSource: {LISTINGS_URL}\n"
        html = f"<p>Check on {ts}</p><p>No new 2 or 3 bed units on 7th floor currently available.</p><p>Source: <a href=\"{LISTINGS_URL}\">{LISTINGS_URL}</a></p>"
    return subject, text, html


def send_email(subject, text, html):
    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"]    = GMAIL_EMAIL
    msg["To"]      = ", ".join(RECIPIENTS)
    msg.attach(MIMEText(text, "plain"))
    msg.attach(MIMEText(html, "html"))
    with smtplib.SMTP("smtp.gmail.com", 587, timeout=30) as smtp:
        smtp.ehlo(); smtp.starttls(); smtp.ehlo()
        smtp.login(GMAIL_EMAIL, GMAIL_APP_PASSWORD.replace(" ", ""))
        smtp.sendmail(GMAIL_EMAIL, RECIPIENTS, msg.as_string())
    log.info("Email sent to: %s", ", ".join(RECIPIENTS))


def run_once():
    log.info("=== Apartment monitor run starting ===")
    state = load_state()
    seen = set(state.get("notified_units", []))
    try:
        listings = fetch_listings()
    except Exception as e:
        log.error("Could not fetch listings: %s\n%s", e, traceback.format_exc())
        listings = []
    matching = [u for u in listings if u.get("floor") == TARGET_FLOOR and u.get("bedrooms") in TARGET_BEDROOMS]
    log.info("Total listings=%d, matching (floor %d, %s bed)=%d", len(listings), TARGET_FLOOR, sorted(TARGET_BEDROOMS), len(matching))
    new_units = [u for u in matching if unit_key(u) not in seen]
    log.info("Previously notified=%d, new=%d", len(seen), len(new_units))
    subject, text, html = build_email(new_units, matching)
    try:
        send_email(subject, text, html)
    except Exception as e:
        log.error("Email failed: %s", e)
        return False
    state["notified_units"] = sorted({unit_key(u) for u in matching})
    state["last_run"] = datetime.now().isoformat(timespec="seconds")
    save_state(state)
    log.info("=== Run complete ===")
    return True


run_once()

## Cell 4 (optional) — Auto-check every few hours while this tab stays open

Google Colab does **not** run your notebook on a schedule by itself when the tab is closed. There are three ways to get twice-a-day checks:

**Option A — Run this notebook manually whenever you want a check.** Simplest. Open the notebook, click **Runtime → Run all**, done.

**Option B — Leave this tab open and run the scheduler cell below.** It'll check at the next 9 AM ET and 5 PM ET, then keep doing so. Free Colab disconnects idle sessions after ~90 minutes, so this only really works if you'll be active in the tab. Keep the browser tab open and visible.

**Option C — True unattended scheduling.** You need either Colab Pro (which has scheduled notebooks) **or** convert this to a Google Apps Script (which has free built-in triggers). Ask Claude for help if you want Option C; it's the only fully "set it and forget it" route.

Run the cell below only if you want Option B.

In [ ]:
# Option B: scheduler loop. Stop it with Runtime → Interrupt execution.
import time
from datetime import datetime, timedelta, timezone

EASTERN = timezone(timedelta(hours=-5))   # EST. Use -4 between mid-March and early November (EDT).
CHECK_HOURS = [9, 17]                     # 9 AM and 5 PM Eastern

def next_run_at(now_et):
    today_targets = [now_et.replace(hour=h, minute=0, second=0, microsecond=0) for h in CHECK_HOURS]
    upcoming = [t for t in today_targets if t > now_et]
    if upcoming:
        return min(upcoming)
    tomorrow = now_et + timedelta(days=1)
    return tomorrow.replace(hour=CHECK_HOURS[0], minute=0, second=0, microsecond=0)

print("Scheduler started. Press the stop button (or Runtime → Interrupt) to stop.")
while True:
    now = datetime.now(EASTERN)
    target = next_run_at(now)
    wait_seconds = (target - now).total_seconds()
    print(f"Next run at {target.strftime('%Y-%m-%d %I:%M %p ET')} — sleeping {int(wait_seconds//60)} min.")
    # sleep in 60s chunks so Interrupt feels responsive
    while wait_seconds > 0:
        chunk = min(60, wait_seconds)
        time.sleep(chunk)
        wait_seconds -= chunk
    run_once()

---
## Troubleshooting

**No email arrived after Cell 3.**
- Check your spam/junk folder first.
- Look at the log output Cell 3 printed. `SMTPAuthenticationError` = the Gmail App Password is wrong. Generate a fresh one at <https://myaccount.google.com/apppasswords> and re-paste it into Cell 1.
- Make sure you used an **App Password**, not your regular Gmail password.

**"Fetched 0 units" in the log.**
- Modera may have changed their site. Open the listings page in a browser to confirm units are showing. The script still emails you ("no new units available") so you're not left guessing.

**The scheduler cell stopped overnight.**
- Free Colab disconnects idle sessions. Either keep the tab focused, upgrade to Colab Pro for scheduled notebooks, or switch to a Google Apps Script trigger.

**I want to force-resend notifications for currently listed units.**
- Open your Drive folder `apartment_monitor/state.json` and replace the contents with `{"notified_units": [], "last_run": null}`. Then run Cell 3 again.